# Production Prompt Management

A prompt that works in a notebook is not yet a **production asset**. Applications need **parameterized templates**, **version control**, safe **storage**, rollback when quality drops, and **observability** so teams know which prompt version produced each response.

This notebook closes the **04. Prompt Engineering** track: how to structure prompts like software — templates with variables, semver-style versions, where to store prompts (code vs config vs database), logging and ownership, and lightweight Python patterns you can grow into CI regression tests (building on notebook 09).

Prerequisites: **01–09** in this section and API integration from **03. LLM API Integration**.

---

## 1. Prompt Templates and Variables

### 1.1 Why Templates

Hard-coding prompts inside route handlers leads to duplication, typos, and untracked edits. **Templates** separate **stable instructions** from **runtime data**:

| Part | Example |
|------|---------|
| Template | "Classify this review as {labels}. Review: {review}" |
| Variables | `labels`, `review` injected at request time |
| Metadata | model, temperature, max_tokens, version id |

### 1.2 Templating Options in Python

| Approach | Pros | Cons |
|----------|------|------|
| **`str.format` / f-strings** | Simple, no deps | Easy to break with `{` in user text |
| **Jinja2** | Conditionals, includes, filters | Extra dependency; sandbox user vars |
| **Dedicated prompt classes** | Type hints, validation | More boilerplate |

**Rule:** never interpolate **untrusted user text** into the system prompt without delimiters (notebook 08). User content belongs in named variables with clear boundaries.

### 1.3 Template Structure

A production prompt record often contains:

```yaml
id: support_sentiment_classifier
version: "2.1.0"
system: |
  You classify support email sentiment...
user_template: |
  Email:\n{{ email_body }}\n\nSentiment:
model: gpt-4o-mini
temperature: 0.0
max_tokens: 10
```

Keep **inference settings** next to the prompt so version bumps capture behavior changes holistically.

---

## 2. Versioning and Change Control

### 2.1 Version Identifiers

Use explicit **version strings** on every prompt:

- **Semantic style:** `major.minor.patch` — patch = wording tweak, minor = new examples, major = task change
- **Git SHA:** immutable trace to exact commit
- **Date + slug:** `2025-03-01-neutrals-fewshot`

Log `prompt_id` + `prompt_version` on every LLM request.

### 2.2 Changelog and Promotion

Each version entry should record:

| Field | Example |
|-------|---------|
| Version | `2.1.0` |
| Author | team or PR number |
| Hypothesis | "NEUTRAL few-shot fixes mixed reviews" |
| Metrics | accuracy 75% → 100% on test set (n=4) |
| Decision | promoted / rolled back |

Promotion flow: dev branch → evaluation (notebook 09) → PR review → merge → deploy with default version flag.

### 2.3 Rollback

Store **multiple versions** in config; switch active version via environment variable or feature flag without redeploying code:

```
PROMPT_VERSION=2.0.0  # rollback from 2.1.0
```

Rollback is instant if old prompt text remains in the registry.

---

## 3. Storage Strategies

| Strategy | Best for | Notes |
|----------|----------|-------|
| **Python modules / constants** | Small apps, few prompts | Versioned in git; requires deploy to change |
| **YAML / JSON files** | Medium apps, non-dev editors | Load at startup; hot-reload optional |
| **Database** | Dynamic per-tenant prompts | Audit table; admin UI; cache locally |
| **Prompt platforms** (LangSmith, etc.) | Teams, experiments | Hosted versioning, eval integration |
| **Git only** | Source of truth always | Platforms sync from git in mature setups |

### 3.1 Recommendations by Stage

| Stage | Approach |
|-------|----------|
| Prototype | Constants in repo |
| Production v1 | YAML in repo + env for active version |
| Multi-team | Git + CI eval + optional platform for tracing |

**Secrets never live in prompt files** — API keys stay in `.env` / secret manager.

### 3.2 Multi-Provider Normalization

Your template layer should map to provider APIs (notebook 05):

- OpenAI: `system` message + `user` template
- Claude: `system` param + `messages`
- Gemini: `system_instruction` + `contents`

One internal `PromptSpec` → adapter per provider.

---

## 4. Observability and Team Workflow

### 4.1 What to Log (Per Request)

| Field | Purpose |
|-------|---------|
| `request_id` | Trace across services |
| `prompt_id`, `prompt_version` | Reproduce behavior |
| `model`, `temperature` | Debug quality drift |
| `prompt_tokens`, `completion_tokens` | Cost |
| `latency_ms` | SLA |
| `parse_success` | Format pipeline health |
| Hashed user id / tenant | Segment metrics (not raw PII in logs) |

Avoid logging full prompts/responses with **PII** unless policy allows; sample or redact in production.

### 4.2 Team Workflow

1. **Owner** per prompt family (support, legal, coding)
2. Changes via **PR** with eval results attached
3. **Reviewer** checks safety, scope, metric delta
4. **CI job** runs notebook 09-style regression on golden test set
5. **On-call** can flip `PROMPT_VERSION` to rollback

### 4.3 Drift Monitoring

Offline accuracy does not guarantee live performance. Monitor:

- Parse error rate spike
- User thumbs-down rate
- Average output length change
- New failure clusters from support tickets

Trigger prompt review when metrics cross thresholds.

---

## 5. Demonstration: Prompt Registry in Python

A minimal in-code registry with versions, rendering, and metadata. Replace the placeholder API key before running.

In [ ]:
OPENAI_API_KEY = "sk-proj-placeholder-replace-with-your-real-key"

import os
import time
import uuid
from dataclasses import dataclass, field
from typing import Any

from openai import OpenAI

client = OpenAI(api_key=OPENAI_API_KEY)


@dataclass(frozen=True)
class PromptSpec:
    id: str
    version: str
    system: str
    user_template: str
    model: str = "gpt-4o-mini"
    temperature: float = 0.0
    max_tokens: int = 50
    changelog: str = ""

    def render_user(self, **variables: Any) -> str:
        return self.user_template.format(**variables)


PROMPT_REGISTRY: dict[str, dict[str, PromptSpec]] = {
    "sentiment_classifier": {
        "1.0.0": PromptSpec(
            id="sentiment_classifier",
            version="1.0.0",
            system="Classify review sentiment.",
            user_template="Review: {review}\nSentiment:",
            changelog="Initial vague prompt",
        ),
        "2.0.0": PromptSpec(
            id="sentiment_classifier",
            version="2.0.0",
            system=(
                "Classify sentiment. Reply with exactly one word: "
                "POSITIVE, NEGATIVE, or NEUTRAL."
            ),
            user_template=(
                "Examples:\n"
                "Review: Good price, bad build. -> NEUTRAL\n\n"
                "Review: {review}\nSentiment:"
            ),
            changelog="Explicit format + NEUTRAL few-shot",
        ),
    }
}


def get_prompt(prompt_id: str, version: str | None = None) -> PromptSpec:
    versions = PROMPT_REGISTRY[prompt_id]
    if version is None:
        version = os.getenv("PROMPT_VERSION", max(versions.keys()))
    return versions[version]


active = get_prompt("sentiment_classifier")
print(f"Active: {active.id} v{active.version}")
print(f"Changelog: {active.changelog}")

---

## 6. Demonstration: Render, Call, and Log

Single function ties template rendering, API call, and structured logging.

In [ ]:
VALID = {"POSITIVE", "NEGATIVE", "NEUTRAL"}


def run_prompt(spec: PromptSpec, request_id: str, **variables: Any) -> dict:
    user_content = spec.render_user(**variables)
    start = time.perf_counter()
    response = client.chat.completions.create(
        model=spec.model,
        messages=[
            {"role": "system", "content": spec.system},
            {"role": "user", "content": user_content},
        ],
        temperature=spec.temperature,
        max_tokens=spec.max_tokens,
    )
    latency_ms = int((time.perf_counter() - start) * 1000)
    raw = (response.choices[0].message.content or "").strip().upper()
    label = next((v for v in VALID if v in raw), raw)
    usage = response.usage

    log_record = {
        "request_id": request_id,
        "prompt_id": spec.id,
        "prompt_version": spec.version,
        "model": spec.model,
        "latency_ms": latency_ms,
        "prompt_tokens": usage.prompt_tokens if usage else None,
        "completion_tokens": usage.completion_tokens if usage else None,
        "parse_success": label in VALID,
        "output_label": label,
    }
    return {"log": log_record, "label": label, "raw": raw}


review = "Battery life is amazing but the screen is dim."
result = run_prompt(active, request_id=str(uuid.uuid4()), review=review)
print("Label:", result["label"])
print("Log record:")
for k, v in result["log"].items():
    print(f"  {k}: {v}")

Ship logs to your observability stack (JSON lines, OpenTelemetry, Datadog). The `prompt_version` field makes post-incident replay possible.

---

## 7. Demonstration: Version Rollback via Environment

Simulate rollback by selecting an older registry version.

In [ ]:
review = "Battery life is amazing but the screen is dim."

for ver in ("2.0.0", "1.0.0"):
    spec = get_prompt("sentiment_classifier", version=ver)
    out = run_prompt(spec, request_id="rollback-demo", review=review)
    print(f"v{ver} -> {out['label']} (parse_ok={out['log']['parse_success']})")

In production, set `PROMPT_VERSION=1.0.0` in the environment to roll back without code changes.

---

## 8. Demonstration: YAML-Style Config (In Memory)

Same registry pattern loaded from a dict — mirrors `prompts/sentiment.yaml` in a real repo.

In [ ]:
YAML_STYLE_CONFIG = {
    "prompts": [
        {
            "id": "ticket_summarizer",
            "version": "1.0.0",
            "system": "Summarize support tickets in 2 sentences. Be factual.",
            "user_template": "Ticket:\n{ticket_body}",
            "model": "gpt-4o-mini",
            "temperature": 0.2,
            "max_tokens": 120,
        }
    ]
}


def load_from_config(config: dict) -> dict[str, PromptSpec]:
    loaded = {}
    for row in config["prompts"]:
        spec = PromptSpec(
            id=row["id"],
            version=row["version"],
            system=row["system"],
            user_template=row["user_template"],
            model=row.get("model", "gpt-4o-mini"),
            temperature=row.get("temperature", 0.0),
            max_tokens=row.get("max_tokens", 256),
        )
        loaded[f"{spec.id}:{spec.version}"] = spec
    return loaded


configs = load_from_config(YAML_STYLE_CONFIG)
summarizer = configs["ticket_summarizer:1.0.0"]
ticket = "User cannot login after password reset. Error 403. Account: acme-4421."

summary_result = run_prompt(
    summarizer,
    request_id="yaml-demo",
    ticket_body=ticket,
)
# Summarizer returns prose — show raw output
response = client.chat.completions.create(
    model=summarizer.model,
    messages=[
        {"role": "system", "content": summarizer.system},
        {"role": "user", "content": summarizer.render_user(ticket_body=ticket)},
    ],
    temperature=summarizer.temperature,
    max_tokens=summarizer.max_tokens,
)
print(response.choices[0].message.content)

---

## 9. Demonstration: CI Regression Hook

Minimal test runner you can call from `pytest` or a GitHub Action — reuses notebook 09's test-case idea.

In [ ]:
GOLDEN_TESTS = [
    ("Best purchase ever!", "POSITIVE"),
    ("Terrible quality.", "NEGATIVE"),
    ("Good battery, bad screen.", "NEUTRAL"),
]


def regression_check(spec: PromptSpec, min_accuracy: float = 0.67) -> bool:
    correct = 0
    for review, expected in GOLDEN_TESTS:
        out = run_prompt(spec, request_id="ci", review=review)
        if out["label"] == expected:
            correct += 1
    accuracy = correct / len(GOLDEN_TESTS)
    print(f"{spec.id} v{spec.version}: accuracy={accuracy:.0%} (threshold {min_accuracy:.0%})")
    return accuracy >= min_accuracy


for ver in ("1.0.0", "2.0.0"):
    spec = get_prompt("sentiment_classifier", version=ver)
    passed = regression_check(spec)
    print(f"  -> {'PASS' if passed else 'FAIL'}\n")

Block merges when CI fails. Tie threshold to product requirements — 67% is low; real suites use dozens of cases and higher bars.

---

## 10. FastAPI Integration Sketch

In **03. LLM API Integration** FastAPI apps, inject the registry at startup:

```python
# app/prompts.py — versioned registry
# app/services/chat_service.py
def handle_classify(review: str, request_id: str):
    spec = get_prompt("sentiment_classifier")
    result = run_prompt(spec, request_id=request_id, review=review)
    logger.info("llm_call", extra=result["log"])
    return {"label": result["label"], "prompt_version": spec.version}
```

Expose `prompt_version` in API responses so clients and support can reference it during incidents.

---

## 11. Common Mistakes

| Mistake | Consequence | Fix |
|---------|-------------|-----|
| Prompts only in notebooks | Untracked prod behavior | Registry in git |
| No version in logs | Cannot debug incidents | Log `prompt_version` |
| Editing prod without eval | Regressions | CI + notebook 09 tests |
| User text in system template | Injection risk | Delimiters + user role |
| One giant template file | Merge conflicts | Split by feature |
| Deleting old versions | Rollback impossible | Keep N previous versions |

---

## 12. Section Recap — Prompt Engineering Track

| # | Notebook | Core idea |
|---|----------|-----------|
| 01 | Introduction | PE lifecycle, vague vs specific |
| 02 | Anatomy | Components, delimiters, templates |
| 03 | Zero / Few-shot | In-context examples |
| 04 | Chain-of-thought | Step-by-step reasoning |
| 05 | Roles / system | Personas, provider APIs |
| 06 | Structured output | JSON mode, schemas |
| 07 | Chaining | Decompose multi-step tasks |
| 08 | Advanced patterns | ToT, ReAct, injection |
| 09 | Evaluation | Metrics, A/B, iteration |
| 10 | Production | Templates, versions, ops |

You now have theory and hands-on patterns from first prompt to operated production assets. Next topics in the curriculum: **RAG**, **agents**, and **advanced production** under `04. Generative AI/`.

---

## 13. Summary

Production prompt management treats prompts as **versioned configuration**: parameterized templates, explicit `prompt_id` / `prompt_version`, storage in git-backed YAML or code registries, rollback via environment flags, and structured logging for every LLM call.

Team workflow pairs PR review with **CI regression** on golden test sets. Monitor live metrics for drift beyond offline accuracy.

Demonstrations built a `PromptSpec` registry, render-and-log execution, version rollback, YAML-style loading, and a minimal CI accuracy check — patterns you can extend into your FastAPI LLM services.

This completes **04. Prompt Engineering**.